In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

dt_ps=0.004

# --- load autocorrelation ---


path="trj/correlation/unified_species/"
df = pd.read_csv( path+"autocorr.csv")
# species signals
C = {
    "dissociated_OH": df["C_dissociated_OH"].values,
    "remaining_OH": df["C_remaining_OH"].values,
    #"C_vac_water_down": df["C_vac_water_down"].values,
    #"C_vac_water_side": df["C_vac_water_side"].values,
    #"C_vac_water_up": df["C_vac_water_up"].values,
    "C_vac_water": df["C_vac_water"].values,
}





def estimate_plateau_excluding_tail(C, exclude_tail_frac=0.5, plateau_frac=0.2):
    """
    Estimate plateau by:
    - discarding the last exclude_tail_frac of the data entirely
    - averaging the last plateau_frac of the *remaining* part
    """

    N = len(C)
    cut = int((1 - exclude_tail_frac) * N)     # endpoint of usable data

    usable = C[:cut]                           # first (1-excl) portion

    start = int((1 - plateau_frac) * len(usable))  # start of plateau window
    plateau_window = usable[start:]

    p = np.mean(plateau_window)
    return p, usable


def subtract_plateau(C,exclude_tail_frac, plateau_frac):
    p, C_cut = estimate_plateau_excluding_tail(C, exclude_tail_frac=0.5, plateau_frac=0.1)
    C_corr = C_cut - p
    C_corr /= 1-p
    return C_corr, p



# --- exponential model ---
def exp1(t, A, tau):
    return A * np.exp(-t / tau)

def fit_and_plot_species(name, t, C_t, t_fit_max=500/dt_ps):
    """
    Fit C(t) up to t_fit_max ps and plot both data + exponential fit.
    """
    # === choose fit window ===
    mask = (t > 0) & (t < t_fit_max)
    t_fit = t[mask]
    C_fit = C_t[mask]

    # === fit ===
    try:
        popt, pcov = curve_fit(exp1, t_fit, C_fit, p0=(1.0, 1.0), maxfev=5000)
        A_fit, tau_fit = popt
    except RuntimeError:
        print(f"Fit failed for {name}")
        return
    
    print(f"{name}: tau_fit = {tau_fit:.4f} ps   (A={A_fit:.3f})")

    # === generate smooth curve ===
    t_smooth = np.linspace(0, t.max(), 800)
    C_smooth = exp1(t_smooth, A_fit, tau_fit)

    # === plot ===
    plt.figure(figsize=(7,5))
    plt.plot(t, C_t, 'o', ms=3, alpha=0.5, label="data")
    plt.plot(t_smooth, C_smooth, 'r-', lw=2, label=f"fit (tau={tau_fit:.3f} ps)")
    
    
    #plt.yscale("log")
    plt.xlabel("τ (ps)")
    plt.ylabel("C(τ)")
    plt.title(f"Autocorrelation: {name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [ ]:
# time spacing: 4 fs = 0.004 ps
dt_ps = 0.004
t =np.arange(len(df)) * dt_ps

N_frame=len(df["C_dissociated_OH"].values)

print('N_frame=',N_frame)
print('time=', N_frame* dt_ps)

In [ ]:
# time spacing: 4 fs = 0.004 ps
dt_ps = 0.004
t = np.arange(len(df)) * dt_ps

# --- identify all correlation columns automatically ---
corr_cols = [col for col in df.columns if col.startswith("C_")]

plt.figure(figsize=(10,6))

for col in corr_cols:
    
    plt.plot(t, df[col].values, label=col, alpha=0.8)

plt.xlabel("Time (ps)")
plt.ylabel("C(t)")
plt.title("Autocorrelation functions (raw)")
plt.legend()
plt.grid(alpha=0.3)

plt.xlim()
plt.tight_layout()
plt.show()

In [ ]:
C_corr = {}
plateaus = {}
exclude_tail_frac=0.5
plateau_frac=0.1
for sp in C.keys():
    
    C_corr[sp], plateaus[sp] = subtract_plateau(C[sp],exclude_tail_frac, plateau_frac)

In [ ]:
for col in C_corr.keys():
    plt.plot(t[:int(len(t)*exclude_tail_frac)], C_corr[col], label=col, alpha=0.8)

plt.xlabel("Time (ps)")
plt.ylabel("C(t)")
plt.title("Autocorrelation delta functions")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
for name, C_t in C_corr.items():
    fit_and_plot_species(name, t[:int(len(t)*(1-exclude_tail_frac))], C_t, t_fit_max=40.0)  # adjust 2.0 ps if needed


In [ ]:
C_corr

In [ ]:
for name, C_t in C_corr.items():
    fit_and_plot_species(name, t, C_t, t_fit_max=10.0)  # adjust 2.0 ps if needed
